# LeetCode 516 — Longest Palindromic Subsequence bằng Python

Link đề: https://leetcode.com/problems/longest-palindromic-subsequence/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming (quy hoạch động theo đoạn / interval DP)

Skeleton LeetCode:

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài** của **dãy con (subsequence)** đối xứng (palindrome) dài nhất của chuỗi `s`, và phân biệt rõ **dãy con** (được bỏ ký tự, giữ thứ tự) với **chuỗi con liên tiếp (substring)**.
2. Nắm các công cụ Python cần dùng: cắt lát và đảo chuỗi `s[::-1]`, tạo bảng 2 chiều đúng cách `[[0] * n for _ in range(n)]`, `functools.lru_cache`, `range(n - 1, -1, -1)` để lặp ngược, và duyệt mọi tập con bằng bitmask.
3. Đi từ **vét cạn mọi dãy con (2^n)** → **đệ quy theo đoạn `[i, j]`** → **ghi nhớ (memo)** → **bảng DP 2 chiều bottom-up** → **tối ưu bộ nhớ còn một mảng** → **đưa về LCS của `s` và `s` đảo ngược**.
4. Hiểu mấu chốt: gọi `f[i][j]` là đáp án cho đoạn `s[i..j]`; nếu hai đầu **bằng nhau** thì "ăn" cả hai và cộng 2 cho đoạn bên trong, nếu **khác nhau** thì bỏ một đầu và lấy tốt hơn trong hai cách.
5. Tránh các lỗi kinh điển: nhầm dãy con với chuỗi con liên tiếp, lặp sai thứ tự `i`/`j` khiến đọc ô chưa tính, mất giá trị đường chéo khi nén về 1 chiều, và tạo bảng bằng `[[0] * n] * n`.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, hãy trả về **độ dài của dãy con đối xứng dài nhất** trong `s`.

**Dãy con (subsequence)** là dãy nhận được từ chuỗi gốc bằng cách **xóa một số (có thể không xóa, có thể xóa nhiều) ký tự**, và **giữ nguyên thứ tự** của các ký tự còn lại. Các ký tự giữ lại **không cần nằm sát nhau**.

**Ví dụ 1**

- Input: `s = "bbbab"`
- Output: `4`
- Giải thích: một dãy con đối xứng dài nhất là `"bbbb"` (giữ 4 chữ `b`, bỏ chữ `a`).

**Ví dụ 2**

- Input: `s = "cbbd"`
- Output: `2`
- Giải thích: một dãy con đối xứng dài nhất là `"bb"`.

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `s` chỉ gồm các chữ cái tiếng Anh viết thường.

Chú ý: đề chỉ hỏi **độ dài**, không hỏi chính dãy con đó. Có thể có nhiều dãy con khác nhau cùng độ dài tối đa, nhưng độ dài là duy nhất.

In [1]:
from typing import List
from functools import lru_cache
from itertools import combinations
import random
import sys
import time

examples = [
    ("bbbab", 4),
    ("cbbd", 2),
]
for s, expected in examples:
    print(repr(s), "len =", len(s), "-> kỳ vọng", expected)

'bbbab' len = 5 -> kỳ vọng 4
'cbbd' len = 4 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác chuỗi con liên tiếp (substring)

Với `s = "bbbab"`:

- **Chuỗi con liên tiếp (substring)**: cắt một đoạn liền nhau, ví dụ `"bba"`, `"bab"`.
- **Dãy con (subsequence)**: chọn một số vị trí tăng dần, ví dụ vị trí `0, 1, 2, 4` cho `"bbbb"`; các vị trí bỏ ở giữa là được.

Mọi substring đều là subsequence, nhưng **ngược lại thì không**. Bài này hỏi subsequence nên "nhảy cóc" được.

In [2]:
s = "bbbab"
n = len(s)

# Dãy con ứng với tập chỉ số tăng dần: dùng combinations để liệt kê các tập chỉ số
idx = (0, 1, 2, 4)
print("dãy con với chỉ số", idx, "=", "".join(s[i] for i in idx))

# Số lượng dãy con (kể cả rỗng) là 2^n; số substring chỉ cỡ n(n+1)/2
print("số tập chỉ số (dãy con) =", 2 ** n, "| số substring không rỗng =", n * (n + 1) // 2)

# Các dãy con có đúng 3 ký tự (theo chỉ số)
subs3 = ["".join(s[i] for i in c) for c in combinations(range(n), 3)]
print(subs3)

dãy con với chỉ số (0, 1, 2, 4) = bbbb
số tập chỉ số (dãy con) = 32 | số substring không rỗng = 15
['bbb', 'bba', 'bbb', 'bba', 'bbb', 'bab', 'bba', 'bbb', 'bab', 'bab']


### 3.2. Chuỗi đối xứng (palindrome) và cách kiểm tra

Palindrome là chuỗi đọc xuôi hay ngược đều như nhau: `"abba"`, `"aba"`, `"b"`, và cả chuỗi rỗng. Python kiểm tra gọn: `t == t[::-1]` (`[::-1]` là bước nhảy `-1`, tức đảo chuỗi). Cách hai con trỏ: so `t[l]` với `t[r]`, rồi `l += 1`, `r -= 1`.

In [3]:
def is_pal(t):
    return t == t[::-1]

def is_pal_two_pointers(t):
    l, r = 0, len(t) - 1
    while l < r:
        if t[l] != t[r]:
            return False
        l += 1
        r -= 1
    return True

for t in ["abba", "aba", "b", "", "abca"]:
    assert is_pal(t) == is_pal_two_pointers(t)
    print(repr(t), "->", is_pal(t))

'abba' -> True
'aba' -> True
'b' -> True
'' -> True
'abca' -> False


### 3.3. Duyệt mọi tập con bằng bitmask

Một chuỗi dài `n` có `2^n` tập chỉ số. Ta đánh số tập con bằng một số nguyên `mask` từ `0` đến `2^n - 1`: bit thứ `i` của `mask` bằng 1 nghĩa là **giữ** ký tự `s[i]`. Biểu thức `(mask >> i) & 1` lấy bit thứ `i`.

In [4]:
s = "abc"
n = len(s)
for mask in range(1 << n):
    sub = "".join(s[i] for i in range(n) if (mask >> i) & 1)
    print(format(mask, "03b")[::-1], "->", repr(sub))

000 -> ''
100 -> 'a'
010 -> 'b'
110 -> 'ab'
001 -> 'c'
101 -> 'ac'
011 -> 'bc'
111 -> 'abc'


### 3.4. Bảng 2 chiều đúng cách (và cái bẫy `[[0] * n] * n`)

Bài này cần bảng `f` cỡ `n x n`. Viết `[[0] * n] * n` là **sai**: nó lặp lại **cùng một list con** `n` lần nên sửa một dòng là sửa mọi dòng. Viết đúng: `[[0] * n for _ in range(n)]` — mỗi vòng tạo một list mới.

In [5]:
bad = [[0] * 3] * 3
bad[0][0] = 7
print("sai :", bad)             # cả ba dòng cùng đổi

good = [[0] * 3 for _ in range(3)]
good[0][0] = 7
print("đúng:", good)            # chỉ dòng 0 đổi

sai : [[7, 0, 0], [7, 0, 0], [7, 0, 0]]
đúng: [[7, 0, 0], [0, 0, 0], [0, 0, 0]]


### 3.5. `functools.lru_cache` và giới hạn đệ quy

`@lru_cache(maxsize=None)` đặt trước một hàm thuần túy (cùng đầu vào luôn cùng kết quả) để **nhớ** kết quả từng lời gọi: lần sau gọi lại cùng tham số sẽ trả ngay, không tính lại. Đệ quy của ta sâu tới khoảng `n` (tối đa 1000), gần ngưỡng mặc định `1000` của Python, nên ta nâng `sys.setrecursionlimit` cho yên tâm.

In [6]:
@lru_cache(maxsize=None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(40), "| số ô đã nhớ:", fib.cache_info().currsize)
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
sys.setrecursionlimit(10000)
print("sau khi nâng:", sys.getrecursionlimit())

102334155 | số ô đã nhớ: 41
giới hạn đệ quy mặc định: 1000
sau khi nâng: 10000


### 3.6. Lặp ngược với `range(n - 1, -1, -1)`

`range(start, stop, step)` với `step = -1` đếm lùi, và `stop` **không** được lấy. Muốn đi từ `n-1` xuống `0` thì `stop = -1`. Thứ tự lặp rất quan trọng trong DP: ta phải tính các ô mà ô hiện tại **phụ thuộc vào** trước.

In [7]:
n = 5
print(list(range(n - 1, -1, -1)))     # i từ n-1 xuống 0
print(list(range(2, n)))              # j từ 2 lên n-1

[4, 3, 2, 1, 0]
[2, 3, 4]


### 3.7. LCS — Longest Common Subsequence (cần cho cách 5)

**LCS** của hai chuỗi `a`, `b` là độ dài dãy con chung dài nhất. Bảng `g[i][j]` = LCS của `a[:i]` và `b[:j]`: nếu `a[i-1] == b[j-1]` thì `g[i][j] = g[i-1][j-1] + 1`, ngược lại `g[i][j] = max(g[i-1][j], g[i][j-1])`. Ví dụ nhỏ dưới đây dùng để làm quen; ta sẽ quay lại ở cách 5.

In [8]:
def lcs(a, b):
    m, k = len(a), len(b)
    g = [[0] * (k + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, k + 1):
            if a[i - 1] == b[j - 1]:
                g[i][j] = g[i - 1][j - 1] + 1
            else:
                g[i][j] = max(g[i - 1][j], g[i][j - 1])
    return g[m][k]

print(lcs("abcde", "ace"))       # "ace" -> 3
print(lcs("abc", "def"))         # 0

3
0


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `s = "bbbab"`. Nhìn **hai đầu** của chuỗi: đầu trái là `b` (vị trí 0), đầu phải là `b` (vị trí 4). Chúng **bằng nhau**.

- Nếu một palindrome dùng hai đầu này làm hai đầu của nó, thì **phần giữa** của palindrome phải là một palindrome nằm trong `s[1..3] = "bba"`.
- Palindrome dài nhất trong `"bba"` là `"bb"` (độ dài 2). Bọc hai đầu `b ... b` vào: `"b" + "bb" + "b" = "bbbb"` → độ dài `2 + 2 = 4`.

Giờ xét `s = "cbbd"`: hai đầu là `c` và `d`, **khác nhau**. Một palindrome không thể dùng cả hai đầu cùng lúc (vì hai đầu của nó phải bằng nhau). Nên ít nhất một trong hai bị **bỏ**:

- bỏ `c`: tìm trong `"bbd"` → `2` (`"bb"`);
- bỏ `d`: tìm trong `"cbb"` → `2` (`"bb"`);
- đáp án = `max(2, 2) = 2`.

Cả hai trường hợp đều rút gọn bài toán thành **cùng bài toán trên một đoạn ngắn hơn**. Đó là dấu hiệu của quy hoạch động theo đoạn. Hãy xem cây gọi đệ quy thật sự: hàm dưới in từng lời gọi với độ thụt lề theo độ sâu.

In [9]:
def lps_show(s, i, j, depth=0):
    pad = "  " * depth
    if i > j:
        print(f"{pad}[{i},{j}] đoạn rỗng -> 0")
        return 0
    if i == j:
        print(f"{pad}[{i},{j}] '{s[i]}' -> 1")
        return 1
    if s[i] == s[j]:
        print(f"{pad}[{i},{j}] '{s[i:j+1]}' hai đầu '{s[i]}' bằng nhau -> 2 + đoạn trong")
        r = 2 + lps_show(s, i + 1, j - 1, depth + 1)
    else:
        print(f"{pad}[{i},{j}] '{s[i:j+1]}' hai đầu '{s[i]}' != '{s[j]}' -> max(bỏ trái, bỏ phải)")
        a = lps_show(s, i + 1, j, depth + 1)
        b = lps_show(s, i, j - 1, depth + 1)
        r = max(a, b)
    print(f"{pad}=> [{i},{j}] = {r}")
    return r

print("KẾT QUẢ:", lps_show("cbbd", 0, 3))

[0,3] 'cbbd' hai đầu 'c' != 'd' -> max(bỏ trái, bỏ phải)
  [1,3] 'bbd' hai đầu 'b' != 'd' -> max(bỏ trái, bỏ phải)
    [2,3] 'bd' hai đầu 'b' != 'd' -> max(bỏ trái, bỏ phải)
      [3,3] 'd' -> 1
      [2,2] 'b' -> 1
    => [2,3] = 1
    [1,2] 'bb' hai đầu 'b' bằng nhau -> 2 + đoạn trong
      [2,1] đoạn rỗng -> 0
    => [1,2] = 2
  => [1,3] = 2
  [0,2] 'cbb' hai đầu 'c' != 'b' -> max(bỏ trái, bỏ phải)
    [1,2] 'bb' hai đầu 'b' bằng nhau -> 2 + đoạn trong
      [2,1] đoạn rỗng -> 0
    => [1,2] = 2
    [0,1] 'cb' hai đầu 'c' != 'b' -> max(bỏ trái, bỏ phải)
      [1,1] 'b' -> 1
      [0,0] 'c' -> 1
    => [0,1] = 1
  => [0,2] = 2
=> [0,3] = 2
KẾT QUẢ: 2


Để ý: cây gọi này sinh ra nhiều lời gọi **trùng nhau** (ví dụ đoạn `[1,2]` xuất hiện ở hai nhánh khác nhau). Với chuỗi dài, sự trùng lặp này là cái ta sẽ khai thác ở các phần sau.

## 5. Cách 1 — Brute Force: thử mọi dãy con

**Ý tưởng:** có `2^n` tập chỉ số. Với mỗi tập (dùng bitmask), ghép các ký tự thành một dãy con, kiểm tra có đối xứng không, và nhớ độ dài lớn nhất.

Rất dễ hiểu và **chắc chắn đúng** nên ta dùng nó làm "trọng tài" (oracle) để kiểm tra các cách nhanh hơn.

In [10]:
def lps_bruteforce(s: str) -> int:
    n = len(s)
    best = 0
    for mask in range(1 << n):                       # mỗi mask là một tập chỉ số được giữ
        sub = [s[i] for i in range(n) if (mask >> i) & 1]
        if len(sub) > best and sub == sub[::-1]:     # chỉ kiểm tra khi có thể cải thiện
            best = len(sub)
    return best

for s, expected in examples:
    got = lps_bruteforce(s)
    print(repr(s), got, "OK" if got == expected else "SAI")

'bbbab' 4 OK
'cbbd' 2 OK


**Độ phức tạp:** `2^n` tập, mỗi tập tốn `O(n)` để dựng và kiểm tra → `O(n · 2^n)` thời gian, `O(n)` bộ nhớ phụ.

**Nút thắt:** `n` tới 1000 thì `2^1000` là con số không tưởng. Đo thử để thấy thời gian tăng gấp đôi mỗi khi `n` tăng 1.

In [11]:
rng = random.Random(1)
for n in [10, 12, 14, 16]:
    s = "".join(rng.choice("abc") for _ in range(n))
    t0 = time.perf_counter()
    r = lps_bruteforce(s)
    dt = time.perf_counter() - t0
    print(f"n = {n:2d}  kết quả {r}  ~{dt * 1000:8.1f} ms")

n = 10  kết quả 6  ~     0.6 ms
n = 12  kết quả 7  ~     2.9 ms
n = 14  kết quả 10  ~    12.7 ms
n = 16  kết quả 11  ~    54.7 ms


## 6. Ý tưởng thuật toán / bước đột phá

Trực giác ở mục 4 cho ta **định nghĩa trạng thái**:

> `f[i][j]` = độ dài dãy con đối xứng dài nhất của đoạn `s[i..j]` (cả hai đầu mút đều tính).

**Trường hợp cơ sở:**

- `i > j` (đoạn rỗng): `0`.
- `i == j` (một ký tự): `1` (một ký tự luôn là palindrome).

**Chuyển trạng thái** (với `i < j`):

- Nếu `s[i] == s[j]`: `f[i][j] = f[i+1][j-1] + 2`.
- Ngược lại: `f[i][j] = max(f[i+1][j], f[i][j-1])`.

**Đáp án:** `f[0][n-1]`.

**Thứ tự tính:** ô `(i, j)` phụ thuộc `(i+1, j-1)`, `(i+1, j)`, `(i, j-1)` — tức là các ô ở **hàng dưới** (`i` lớn hơn) hoặc **cột bên trái** (`j` nhỏ hơn). Vì vậy ta lặp `i` **từ lớn xuống nhỏ** và `j` **từ nhỏ lên lớn** (`j > i`).

Tính tay bảng cho `s = "bbbab"` (chỉ số 0..4):

| `i \ j` | 0 `b` | 1 `b` | 2 `b` | 3 `a` | 4 `b` |
|---|---|---|---|---|---|
| 0 `b` | 1 | 2 | 3 | 3 | **4** |
| 1 `b` | | 1 | 2 | 2 | 3 |
| 2 `b` | | | 1 | 1 | 3 |
| 3 `a` | | | | 1 | 1 |
| 4 `b` | | | | | 1 |

Ví dụ ô `(0,4)`: `s[0] = s[4] = 'b'` nên `f = f[1][3] + 2 = 2 + 2 = 4`. Ô `(2,4)`: `'b' == 'b'` nên `f = f[3][3] + 2 = 1 + 2 = 3` (dãy `"bab"`). Ô `(2,3)`: `'b' != 'a'` nên `max(f[3][3], f[2][2]) = max(1, 1) = 1`. Có `n²/2` ô, mỗi ô tính `O(1)` ⇒ `O(n²)`, tốt hơn rất nhiều so với `2^n`.

## 7. Các cách tiếp theo

Từ cùng một công thức, ta có ba cách cài đặt khác nhau về hình thức (đệ quy + nhớ, bảng 2 chiều, một mảng) cộng thêm một cách có góc nhìn **hoàn toàn khác** (đưa về LCS). Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Đệ quy theo đoạn + ghi nhớ (top-down memo)

**Giải thích từng đoạn:**

1. Hàm `dfs(i, j)` trả về đáp án cho đoạn `s[i..j]`.
2. Nếu `i > j` trả `0`; nếu `i == j` trả `1` (hai điểm dừng).
3. Nếu hai đầu bằng nhau: `2 + dfs(i+1, j-1)`.
4. Nếu khác: `max(dfs(i+1, j), dfs(i, j-1))`.
5. `@lru_cache` ghi nhớ để mỗi cặp `(i, j)` chỉ tính một lần ⇒ tối đa `n²` trạng thái.

Trước hết xem **không nhớ** thì số lời gọi bùng nổ ra sao (chuỗi toàn ký tự khác nhau là trường hợp xấu: luôn rẽ hai nhánh).

In [12]:
def count_calls_without_memo(s):
    calls = 0
    def dfs(i, j):
        nonlocal calls
        calls += 1
        if i > j:
            return 0
        if i == j:
            return 1
        if s[i] == s[j]:
            return 2 + dfs(i + 1, j - 1)
        return max(dfs(i + 1, j), dfs(i, j - 1))
    ans = dfs(0, len(s) - 1)
    return ans, calls

def count_calls_with_memo(s):
    calls = 0
    @lru_cache(maxsize=None)
    def dfs(i, j):
        nonlocal calls
        calls += 1                       # chỉ đếm khi thật sự tính (cache miss)
        if i > j:
            return 0
        if i == j:
            return 1
        if s[i] == s[j]:
            return 2 + dfs(i + 1, j - 1)
        return max(dfs(i + 1, j), dfs(i, j - 1))
    ans = dfs(0, len(s) - 1)
    return ans, calls

for n in [6, 10, 14, 18]:
    s = "abcdefghijklmnopqrstuvwxyz"[:n]          # toàn ký tự khác nhau
    a, c1 = count_calls_without_memo(s)
    b, c2 = count_calls_with_memo(s)
    assert a == b == 1
    print(f"n = {n:2d}: không nhớ {c1:7d} lời gọi | có nhớ {c2:4d} lần tính")

n =  6: không nhớ      63 lời gọi | có nhớ   21 lần tính
n = 10: không nhớ    1023 lời gọi | có nhớ   55 lần tính
n = 14: không nhớ   16383 lời gọi | có nhớ  105 lần tính
n = 18: không nhớ  262143 lời gọi | có nhớ  171 lần tính


In [13]:
class SolutionMemo:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 5000))

        @lru_cache(maxsize=None)
        def dfs(i: int, j: int) -> int:
            if i > j:
                return 0
            if i == j:
                return 1
            if s[i] == s[j]:
                return dfs(i + 1, j - 1) + 2
            return max(dfs(i + 1, j), dfs(i, j - 1))

        return dfs(0, n - 1)

for s, expected in examples:
    print(repr(s), SolutionMemo().longestPalindromeSubseq(s), "kỳ vọng", expected)

'bbbab' 4 kỳ vọng 4
'cbbd' 2 kỳ vọng 2


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)

        @cache
        def dfs(i: int, j: int) -> int:
            if i > j:
                return 0
            if i == j:
                return 1
            if s[i] == s[j]:
                return dfs(i + 1, j - 1) + 2
            return max(dfs(i + 1, j), dfs(i, j - 1))

        return dfs(0, n - 1)
```

**Độ phức tạp:** `O(n²)` trạng thái, mỗi trạng thái `O(1)` ⇒ `O(n²)` thời gian; bộ nhớ `O(n²)` cho cache cộng `O(n)` ngăn xếp đệ quy. (Trên LeetCode `@cache` có sẵn nhờ `from functools import cache`.)

### 7.2. Cách 3 — Bảng DP 2 chiều bottom-up

**Giải thích từng đoạn:**

1. Tạo bảng `f` cỡ `n x n` toàn `0`.
2. Đặt `f[i][i] = 1` cho mọi `i` (một ký tự).
3. Lặp `i` từ `n-1` xuống `0`, với mỗi `i` lặp `j` từ `i+1` lên `n-1`.
4. Nếu `s[i] == s[j]` thì `f[i][j] = f[i+1][j-1] + 2`, ngược lại `max(f[i+1][j], f[i][j-1])`.
5. Trả `f[0][n-1]`.

Lưu ý khi `j = i + 1`: ô `f[i+1][j-1] = f[i+1][i]` nằm **dưới đường chéo**, luôn là `0` — đúng với "đoạn rỗng".

In [14]:
class SolutionTable:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        f = [[0] * n for _ in range(n)]
        for i in range(n):
            f[i][i] = 1
        for i in range(n - 1, -1, -1):
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    f[i][j] = f[i + 1][j - 1] + 2
                else:
                    f[i][j] = max(f[i + 1][j], f[i][j - 1])
        return f[0][n - 1]

for s, expected in examples:
    print(repr(s), SolutionTable().longestPalindromeSubseq(s), "kỳ vọng", expected)

'bbbab' 4 kỳ vọng 4
'cbbd' 2 kỳ vọng 2


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        f = [[0] * n for _ in range(n)]
        for i in range(n):
            f[i][i] = 1
        for i in range(n - 1, -1, -1):
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    f[i][j] = f[i + 1][j - 1] + 2
                else:
                    f[i][j] = max(f[i + 1][j], f[i][j - 1])
        return f[0][n - 1]
```

**Độ phức tạp:** thời gian `O(n²)`, bộ nhớ `O(n²)`. Không đệ quy nên không lo tràn ngăn xếp.

### 7.3. Cách 4 — Nén về một mảng (rolling array)

**Quan sát:** hàng `i` của bảng chỉ đọc **hàng `i+1`** (hàng ngay dưới). Nên không cần cả bảng: ta giữ **một mảng `dp` dài `n`** thể hiện hàng đang xử lý. Khi chuyển sang hàng `i` mới, ta **ghi đè** dần `dp[j]`.

**Giải thích từng đoạn:**

1. `dp[j]` trước khi cập nhật chính là `f[i+1][j]` (ô "bên dưới"); sau khi cập nhật là `f[i][j]`.
2. Ô `f[i+1][j-1]` (đường chéo) là giá trị `dp[j-1]` **của hàng cũ** — nhưng khi lặp `j` tăng dần, `dp[j-1]` đã bị ghi đè thành `f[i][j-1]`. Vì vậy ta phải **cất** nó vào biến `prev` trước khi ghi đè.
3. Ô `f[i][j-1]` (bên trái) chính là `dp[j-1]` **mới** (đã cập nhật ở vòng `j-1`).
4. Với mỗi hàng `i`: đặt `dp[i] = 1`; biến `prev` giữ `f[i+1][j-1]` — khởi tạo bằng `dp[i]` cũ trước khi gán là hàng cũ (`f[i+1][i]`, chính là `0`).

In [15]:
class SolutionRolling:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        dp = [0] * n
        for i in range(n - 1, -1, -1):
            prev = 0                    # f[i+1][i] = 0 (đoạn rỗng); sẽ là đường chéo f[i+1][j-1]
            dp[i] = 1                   # f[i][i]
            for j in range(i + 1, n):
                tmp = dp[j]             # f[i+1][j]: giữ lại, vì sẽ thành "đường chéo" của ô kế tiếp
                if s[i] == s[j]:
                    dp[j] = prev + 2
                else:
                    dp[j] = max(dp[j], dp[j - 1])   # max(f[i+1][j], f[i][j-1])
                prev = tmp
        return dp[n - 1]

for s, expected in examples:
    print(repr(s), SolutionRolling().longestPalindromeSubseq(s), "kỳ vọng", expected)

'bbbab' 4 kỳ vọng 4
'cbbd' 2 kỳ vọng 2


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        dp = [0] * n
        for i in range(n - 1, -1, -1):
            prev = 0
            dp[i] = 1
            for j in range(i + 1, n):
                tmp = dp[j]
                if s[i] == s[j]:
                    dp[j] = prev + 2
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return dp[n - 1]
```

**Độ phức tạp:** thời gian `O(n²)`, bộ nhớ **`O(n)`**.

### 7.4. Cách 5 — Đưa về LCS: `s` và `s` đảo ngược

**Góc nhìn khác:** một palindrome đọc xuôi và ngược như nhau. Nên một **dãy con đối xứng** của `s` vừa là dãy con của `s`, vừa là dãy con của `s[::-1]`. Ngược lại, **LCS** của `s` và `s[::-1]` là một dãy con chung; người ta chứng minh được ta luôn có thể chọn nó đối xứng (nếu chưa thì "gập" lại để có độ dài không giảm). Kết luận:

> độ dài dãy con đối xứng dài nhất của `s` = `LCS(s, s[::-1])`.

Cách này không dùng "đoạn `[i, j]`" mà dùng **hai tiền tố** `i`, `j` của `s` và `t = s[::-1]`. Ta tái sử dụng khung LCS đã học ở mục 3.7 — chỉ cần viết lại gọn và dùng hai mảng cuộn (độ phức tạp bộ nhớ `O(n)`).

In [16]:
class SolutionLCS:
    def longestPalindromeSubseq(self, s: str) -> int:
        t = s[::-1]
        n = len(s)
        prev = [0] * (n + 1)            # hàng i-1 của bảng LCS
        for i in range(1, n + 1):
            cur = [0] * (n + 1)         # hàng i
            for j in range(1, n + 1):
                if s[i - 1] == t[j - 1]:
                    cur[j] = prev[j - 1] + 1
                else:
                    cur[j] = max(prev[j], cur[j - 1])
            prev = cur
        return prev[n]

for s, expected in examples:
    print(repr(s), "t =", repr(s[::-1]), SolutionLCS().longestPalindromeSubseq(s), "kỳ vọng", expected)

'bbbab' t = 'babbb' 4 kỳ vọng 4
'cbbd' t = 'dbbc' 2 kỳ vọng 2


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        t = s[::-1]
        n = len(s)
        prev = [0] * (n + 1)
        for i in range(1, n + 1):
            cur = [0] * (n + 1)
            for j in range(1, n + 1):
                if s[i - 1] == t[j - 1]:
                    cur[j] = prev[j - 1] + 1
                else:
                    cur[j] = max(prev[j], cur[j - 1])
            prev = cur
        return prev[n]
```

**Độ phức tạp:** thời gian `O(n²)` (đúng `n·n` ô, nhiều hơn cách 3 khoảng 2 lần), bộ nhớ `O(n)`.

Lưu ý: cách này khác cách 3/4 về **trạng thái** (tiền tố của hai chuỗi, không phải đoạn của một chuỗi) nên rất tiện làm "kiểm tra chéo" độc lập.

## 8. TRACE từng bước

Ta theo dõi cách 3 trên `s = "bbbab"`: với mỗi ô `(i, j)` in ra hai ký tự đầu mút, trường hợp nào được dùng, các ô phụ thuộc, giá trị mới; sau mỗi hàng `i` in toàn bộ bảng.

Bảng trạng thái của toàn bộ 10 bước (kết quả cell trace dưới đây phải khớp):

| bước | `(i, j)` | `s[i]`, `s[j]` | trường hợp | giá trị |
|---|---|---|---|---|
| 1 | (3,4) | `a`, `b` | khác → `max(f[4][4], f[3][3])` | 1 |
| 2 | (2,3) | `b`, `a` | khác → `max(f[3][3], f[2][2])` | 1 |
| 3 | (2,4) | `b`, `b` | bằng → `f[3][3] + 2` | 3 |
| 4 | (1,2) | `b`, `b` | bằng → `f[2][1] + 2` (đoạn rỗng = 0) | 2 |
| 5 | (1,3) | `b`, `a` | khác → `max(f[2][3], f[1][2])` | 2 |
| 6 | (1,4) | `b`, `b` | bằng → `f[2][3] + 2` | 3 |
| 7 | (0,1) | `b`, `b` | bằng → `f[1][0] + 2` (rỗng = 0) | 2 |
| 8 | (0,2) | `b`, `b` | bằng → `f[1][1] + 2` | 3 |
| 9 | (0,3) | `b`, `a` | khác → `max(f[1][3], f[0][2])` | 3 |
| 10 | (0,4) | `b`, `b` | bằng → `f[1][3] + 2` | **4** |

In [17]:
def trace_table(s):
    n = len(s)
    f = [[0] * n for _ in range(n)]
    for i in range(n):
        f[i][i] = 1

    def show(i_now):
        print("    " + " ".join(f"{j}:{s[j]}" for j in range(n)))
        for r in range(n):
            row = []
            for c in range(n):
                row.append(" . " if c < r else f"{f[r][c]:^3d}")
            print(f"{r}:{s[r]} " + " ".join(row) + ("   <- vừa xong" if r == i_now else ""))

    step = 0
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            step += 1
            if s[i] == s[j]:
                f[i][j] = f[i + 1][j - 1] + 2
                why = f"'{s[i]}' == '{s[j]}' -> f[{i+1}][{j-1}] + 2 = {f[i+1][j-1]} + 2"
            else:
                f[i][j] = max(f[i + 1][j], f[i][j - 1])
                why = f"'{s[i]}' != '{s[j]}' -> max(f[{i+1}][{j}], f[{i}][{j-1}]) = max({f[i+1][j]}, {f[i][j-1]})"
            print(f"bước {step:2d}: (i={i}, j={j}) {why} => {f[i][j]}")
        print(f"--- bảng sau khi xong hàng i = {i} ---")
        show(i)
        print()
    return f[0][n - 1]

print("ĐÁP ÁN:", trace_table("bbbab"))

--- bảng sau khi xong hàng i = 4 ---
    0:b 1:b 2:b 3:a 4:b
0:b  1   0   0   0   0 
1:b  .   1   0   0   0 
2:b  .   .   1   0   0 
3:a  .   .   .   1   0 
4:b  .   .   .   .   1    <- vừa xong

bước  1: (i=3, j=4) 'a' != 'b' -> max(f[4][4], f[3][3]) = max(1, 1) => 1
--- bảng sau khi xong hàng i = 3 ---
    0:b 1:b 2:b 3:a 4:b
0:b  1   0   0   0   0 
1:b  .   1   0   0   0 
2:b  .   .   1   0   0 
3:a  .   .   .   1   1    <- vừa xong
4:b  .   .   .   .   1 

bước  2: (i=2, j=3) 'b' != 'a' -> max(f[3][3], f[2][2]) = max(1, 1) => 1
bước  3: (i=2, j=4) 'b' == 'b' -> f[3][3] + 2 = 1 + 2 => 3
--- bảng sau khi xong hàng i = 2 ---
    0:b 1:b 2:b 3:a 4:b
0:b  1   0   0   0   0 
1:b  .   1   0   0   0 
2:b  .   .   1   1   3    <- vừa xong
3:a  .   .   .   1   1 
4:b  .   .   .   .   1 

bước  4: (i=1, j=2) 'b' == 'b' -> f[2][1] + 2 = 0 + 2 => 2
bước  5: (i=1, j=3) 'b' != 'a' -> max(f[2][3], f[1][2]) = max(1, 2) => 2
bước  6: (i=1, j=4) 'b' == 'b' -> f[2][3] + 2 = 1 + 2 => 3
--- bảng sau khi

Bạn có thể thấy: hàng `i = 4` chỉ có một ô (đường chéo `1`); mỗi hàng mới thêm một ô ở chéo và dần "lấp" cả tam giác trên. Ô `(0,4)` ra `4`, khớp với tính tay.

Bây giờ trace cách 4 (một mảng): in `dp`, `prev` và `tmp` ở từng bước để thấy vì sao phải cất đường chéo.

In [18]:
def trace_rolling(s):
    n = len(s)
    dp = [0] * n
    for i in range(n - 1, -1, -1):
        prev = 0
        dp[i] = 1
        print(f"=== hàng i = {i} ('{s[i]}'): đặt dp[{i}] = 1 -> dp = {dp}")
        for j in range(i + 1, n):
            tmp = dp[j]
            if s[i] == s[j]:
                dp[j] = prev + 2
                how = f"bằng nhau: dp[{j}] = prev + 2 = {prev} + 2"
            else:
                dp[j] = max(dp[j], dp[j - 1])
                how = f"khác nhau: dp[{j}] = max(cũ {tmp}, dp[{j-1}] mới {dp[j-1]})"
            print(f"  j = {j} ('{s[j]}'): {how} -> dp = {dp} (prev: {prev} -> {tmp})")
            prev = tmp
    return dp[n - 1]

print("ĐÁP ÁN:", trace_rolling("cbbd"))

=== hàng i = 3 ('d'): đặt dp[3] = 1 -> dp = [0, 0, 0, 1]
=== hàng i = 2 ('b'): đặt dp[2] = 1 -> dp = [0, 0, 1, 1]
  j = 3 ('d'): khác nhau: dp[3] = max(cũ 1, dp[2] mới 1) -> dp = [0, 0, 1, 1] (prev: 0 -> 1)
=== hàng i = 1 ('b'): đặt dp[1] = 1 -> dp = [0, 1, 1, 1]
  j = 2 ('b'): bằng nhau: dp[2] = prev + 2 = 0 + 2 -> dp = [0, 1, 2, 1] (prev: 0 -> 1)
  j = 3 ('d'): khác nhau: dp[3] = max(cũ 1, dp[2] mới 2) -> dp = [0, 1, 2, 2] (prev: 1 -> 1)
=== hàng i = 0 ('c'): đặt dp[0] = 1 -> dp = [1, 1, 2, 2]
  j = 1 ('b'): khác nhau: dp[1] = max(cũ 1, dp[0] mới 1) -> dp = [1, 1, 2, 2] (prev: 0 -> 1)
  j = 2 ('b'): khác nhau: dp[2] = max(cũ 2, dp[1] mới 1) -> dp = [1, 1, 2, 2] (prev: 1 -> 2)
  j = 3 ('d'): khác nhau: dp[3] = max(cũ 2, dp[2] mới 2) -> dp = [1, 1, 2, 2] (prev: 2 -> 2)
ĐÁP ÁN: 2


## 9. Code nộp LeetCode

Ta chọn **Cách 3 (bảng 2 chiều)** làm bản nộp chính vì dễ đọc, không đệ quy, và đủ nhanh cho `n ≤ 1000` (khoảng 500 nghìn ô). Dưới đây là class `Solution` đúng chữ ký của LeetCode:

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        f = [[0] * n for _ in range(n)]
        for i in range(n):
            f[i][i] = 1
        for i in range(n - 1, -1, -1):
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    f[i][j] = f[i + 1][j - 1] + 2
                else:
                    f[i][j] = max(f[i + 1][j], f[i][j - 1])
        return f[0][n - 1]
```

In [19]:
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        f = [[0] * n for _ in range(n)]
        for i in range(n):
            f[i][i] = 1
        for i in range(n - 1, -1, -1):
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    f[i][j] = f[i + 1][j - 1] + 2
                else:
                    f[i][j] = max(f[i + 1][j], f[i][j - 1])
        return f[0][n - 1]

for s, expected in examples:
    got = Solution().longestPalindromeSubseq(s)
    assert got == expected, (s, got, expected)
    print(repr(s), "->", got, "OK")

'bbbab' -> 4 OK
'cbbd' -> 2 OK


## 10. Vì sao đúng

Ta cần chứng minh hai công thức ở mục 6.

**Trường hợp `s[i] != s[j]`.** Gọi `P` là một palindrome con dài nhất của `s[i..j]`. Hai đầu của `P` bằng nhau, mà `s[i] != s[j]` nên `P` **không thể dùng đồng thời** `s[i]` và `s[j]`. Vậy `P` nằm trọn trong `s[i+1..j]` hoặc trong `s[i..j-1]` ⇒ `f[i][j] <= max(f[i+1][j], f[i][j-1])`. Chiều ngược lại hiển nhiên (dãy con của đoạn con cũng là dãy con của đoạn lớn). Suy ra dấu `=`.

**Trường hợp `s[i] == s[j]`.**

- (≥) Lấy palindrome tối ưu `Q` của `s[i+1..j-1]`, bọc thêm `s[i]` ở đầu và `s[j]` ở cuối: được palindrome độ dài `|Q| + 2` ⇒ `f[i][j] >= f[i+1][j-1] + 2`.
- (≤) Lấy palindrome tối ưu `P` của `s[i..j]`. Nếu `|P| = 1` thì `|P| = 1 < f[i+1][j-1] + 2`. Nếu `|P| >= 2`: ký tự đầu của `P` nằm ở chỉ số `l >= i`, ký tự cuối ở chỉ số `r <= j`, `l < r`. Bỏ hai ký tự ngoài cùng của `P`, phần còn lại vẫn đối xứng và nằm trong `s[l+1..r-1]`, mà `s[l+1..r-1]` là đoạn con của `s[i+1..j-1]`. Vậy phần giữa dài tối đa `f[i+1][j-1]` ⇒ `|P| <= f[i+1][j-1] + 2`. (Chú ý bất đẳng thức này đúng với **mọi** `s[i], s[j]`; nhưng chỉ khi `s[i] == s[j]` ta mới thực sự "dựng" được palindrome có hai đầu là `i` và `j`, nên dấu `=` chỉ xảy ra ở nhánh này.)

Hai chiều (≥) và (≤) cho `f[i][j] = f[i+1][j-1] + 2` khi `s[i] == s[j]`.

**Kiểm chứng bằng code:** thay vì tin vào lập luận dài, ta kiểm bất đẳng thức then chốt trên mọi chuỗi nhỏ: khi `s[i] == s[j]` thì `f[i+1][j-1] + 2 >= max(f[i+1][j], f[i][j-1])`, tức là chọn "ăn cả hai đầu" **không bao giờ thua** cách bỏ một đầu.

In [20]:
from itertools import product

def f_table(s):
    n = len(s)
    f = [[0] * (n + 1) for _ in range(n + 1)]    # dư một hàng/cột để chỉ số i+1, j-1 không tràn
    for i in range(n - 1, -1, -1):
        f[i][i] = 1
        for j in range(i + 1, n):
            f[i][j] = f[i + 1][j - 1] + 2 if s[i] == s[j] else max(f[i + 1][j], f[i][j - 1])
    return f

checked = 0
for n in range(2, 9):
    for tup in product("ab", repeat=n):
        s = "".join(tup)
        f = f_table(s)
        for i in range(n):
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    inner = f[i + 1][j - 1] if j - 1 >= i + 1 else 0
                    best_other = max(f[i + 1][j], f[i][j - 1])
                    assert inner + 2 >= best_other, (s, i, j)
                    checked += 1
print("đã kiểm", checked, "trường hợp: 'ăn cả hai đầu' không bao giờ kém hơn bỏ một đầu")

đã kiểm 5630 trường hợp: 'ăn cả hai đầu' không bao giờ kém hơn bỏ một đầu


Kiểm tra trên mọi chuỗi `ab` độ dài ≤ 8 khớp với lập luận: khi hai đầu bằng nhau, "ăn cả hai đầu" luôn không tệ hơn việc bỏ một đầu.

Về chỗ đúng của **bảng**: mỗi ô chỉ phụ thuộc ô ở hàng dưới hoặc cột trái, và ta tính theo thứ tự `i` giảm, `j` tăng nên khi tính ô nào thì các ô nó cần đã sẵn sàng. Số ô hữu hạn nên thuật toán dừng.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force (bitmask) | `O(n · 2^n)` | `O(n)` |
| 2. Memo top-down | `O(n²)` | `O(n²)` (+ ngăn xếp `O(n)`) |
| 3. Bảng 2 chiều | `O(n²)` | `O(n²)` |
| 4. Một mảng | `O(n²)` | `O(n)` |
| 5. LCS với `s[::-1]` | `O(n²)` | `O(n)` |

Với `n = 1000`: khoảng `500.000` ô (cách 3, 4) hoặc `1.000.000` ô (cách 5) — chạy Python trong dưới một giây. Ta đếm trực tiếp số lần cập nhật ô để thấy `~ n²/2`.

In [21]:
def count_cells(n):
    c = 0
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            c += 1
    return c

for n in [10, 100, 200, 1000]:
    c = count_cells(n)
    print(f"n = {n:4d}: số ô cần tính = {c:7d}  | n(n-1)/2 = {n * (n - 1) // 2:7d}")

n =   10: số ô cần tính =      45  | n(n-1)/2 =      45
n =  100: số ô cần tính =    4950  | n(n-1)/2 =    4950
n =  200: số ô cần tính =   19900  | n(n-1)/2 =   19900
n = 1000: số ô cần tính =  499500  | n(n-1)/2 =  499500


## 12. Bộ test

Ta gom tất cả cách vào một dict để so sánh. Các phần: ví dụ đề, edge case, phản ví dụ, đối chiếu với brute force (oracle độc lập), ngẫu nhiên có seed, và kiểm tra tốc độ ở ràng buộc lớn nhất.

In [22]:
methods = {
    "memo": lambda s: SolutionMemo().longestPalindromeSubseq(s),
    "table": lambda s: SolutionTable().longestPalindromeSubseq(s),
    "rolling": lambda s: SolutionRolling().longestPalindromeSubseq(s),
    "lcs": lambda s: SolutionLCS().longestPalindromeSubseq(s),
    "final": lambda s: Solution().longestPalindromeSubseq(s),
}

def check_all(s, expected=None):
    results = {name: fn(s) for name, fn in methods.items()}
    assert len(set(results.values())) == 1, (s, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (s, r, expected)
    return r

# 1) Ví dụ đề
for s, expected in examples:
    check_all(s, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    ("a", 1),                      # n nhỏ nhất
    ("aa", 2),
    ("ab", 1),
    ("aaaaaaaa", 8),               # toàn giống nhau
    ("abcdefg", 1),                # toàn khác nhau
    ("racecar", 7),                # đã là palindrome
    ("abcba", 5),
    ("abcdeabcde", 3),             # chỉ ghép được một cặp trùng + 1 ký tự giữa
]
for s, expected in edge:
    check_all(s, expected)
    if len(s) <= 12:
        assert lps_bruteforce(s) == expected, s
print("edge case: OK")

ví dụ đề: OK
edge case: OK


In [23]:
# 3) Phản ví dụ cho cách hiểu SAI "chuỗi con liên tiếp"
def longest_pal_substring_length(s):
    best = 0
    for c in range(len(s)):
        for l, r in ((c, c), (c, c + 1)):
            while l >= 0 and r < len(s) and s[l] == s[r]:
                l -= 1
                r += 1
            best = max(best, r - l - 1)
    return best

for s in ["bbbab", "cbbd", "agbdba", "abcabcabc"]:
    sub = longest_pal_substring_length(s)
    seq = check_all(s)
    print(f"{s!r:14s} substring liên tiếp = {sub} | dãy con (đề bài) = {seq}", "<- khác nhau!" if sub != seq else "")
assert longest_pal_substring_length("bbbab") == 3 and check_all("bbbab") == 4

'bbbab'        substring liên tiếp = 3 | dãy con (đề bài) = 4 <- khác nhau!
'cbbd'         substring liên tiếp = 2 | dãy con (đề bài) = 2 
'agbdba'       substring liên tiếp = 3 | dãy con (đề bài) = 5 <- khác nhau!
'abcabcabc'    substring liên tiếp = 1 | dãy con (đề bài) = 5 <- khác nhau!


In [24]:
# 4) Đối chiếu với brute force trên mọi chuỗi 'abc' độ dài <= 8 (3^8 = 6561 chuỗi), mỗi cái so 5 cách
count = 0
for n in range(1, 9):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        assert check_all(s) == lps_bruteforce(s), s
        count += 1
print("đã đối chiếu", count, "chuỗi với brute force: OK")

đã đối chiếu 9840 chuỗi với brute force: OK


In [25]:
# 5) Ngẫu nhiên có seed: n <= 12 so brute force; n lớn so giữa các cách DP với nhau
rng = random.Random(2024)
for _ in range(300):
    n = rng.randint(1, 12)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.randint(1, 5)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    assert check_all(s) == lps_bruteforce(s), s
print("ngẫu nhiên nhỏ so brute force: OK")

for _ in range(30):
    n = rng.randint(100, 300)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.randint(1, 26)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    check_all(s)
print("ngẫu nhiên lớn (đối chiếu 5 cách DP): OK")

# 6) Giới hạn lớn nhất n = 1000, ba dạng xấu
cases = {
    "toàn 'a'": "a" * 1000,
    "ngẫu nhiên 26 chữ": "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(1000)),
    "lặp abc": ("abc" * 400)[:1000],
}
for name, s in cases.items():
    t0 = time.perf_counter()
    r = Solution().longestPalindromeSubseq(s)
    dt = time.perf_counter() - t0
    print(f"{name:20s} n = {len(s)} -> {r:4d} ({dt:.2f}s)")
assert Solution().longestPalindromeSubseq("a" * 1000) == 1000

ngẫu nhiên nhỏ so brute force: OK


ngẫu nhiên lớn (đối chiếu 5 cách DP): OK
toàn 'a'             n = 1000 -> 1000 (0.05s)
ngẫu nhiên 26 chữ    n = 1000 ->  319 (0.06s)
lặp abc              n = 1000 ->  667 (0.06s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm dãy con với chuỗi con liên tiếp (mở rộng từ tâm)

Bài 5 (Longest Palindromic **Substring**) giải bằng mở rộng từ tâm; đem nguyên sang bài này là **sai** vì dãy con được "nhảy cóc".

In [26]:
class WrongSubstring:
    def longestPalindromeSubseq(self, s: str) -> int:
        best = 0
        for c in range(len(s)):
            for l, r in ((c, c), (c, c + 1)):
                while l >= 0 and r < len(s) and s[l] == s[r]:
                    l -= 1
                    r += 1
                best = max(best, r - l - 1)
        return best

print("sai :", WrongSubstring().longestPalindromeSubseq("bbbab"), "(đúng phải là 4: 'bbbb' bỏ qua chữ a)")
print("đúng:", Solution().longestPalindromeSubseq("bbbab"))

sai : 3 (đúng phải là 4: 'bbbb' bỏ qua chữ a)
đúng: 4


**Vì sao sai:** mở rộng từ tâm chỉ ghép được các ký tự **sát nhau**; nó dừng ngay khi gặp `a` chen giữa. **Cách sửa:** dùng DP theo đoạn như trên (cho phép "bỏ" ký tự).

### Lỗi 2: lặp sai thứ tự — đọc ô chưa tính

Nếu lặp `i` tăng dần, ô `f[i+1][...]` ở hàng dưới **chưa** được tính (còn `0`), nên công thức dùng giá trị rác.

In [27]:
def wrong_order(s):
    n = len(s)
    f = [[0] * n for _ in range(n)]
    for i in range(n):
        f[i][i] = 1
    for i in range(n - 1):                    # SAI: i tăng dần
        for j in range(i + 1, n):
            if s[i] == s[j]:
                f[i][j] = f[i + 1][j - 1] + 2
            else:
                f[i][j] = max(f[i + 1][j], f[i][j - 1])
    return f[0][n - 1]

print("sai :", wrong_order("bbbab"), "| đúng:", Solution().longestPalindromeSubseq("bbbab"))

sai : 2 | đúng: 4


**Vì sao sai:** khi tính hàng `i = 0`, hàng `i = 1` mới chỉ có đường chéo `1` còn các ô khác là `0`. **Cách sửa:** `for i in range(n - 1, -1, -1)` — tính hàng dưới trước. (Hoặc đảo vai trò: lặp theo **độ dài đoạn** tăng dần.)

### Lỗi 3: nén về một mảng nhưng quên cất đường chéo

Ở cách 4, ghi đè `dp[j]` mất giá trị `f[i+1][j]` mà ô `j+1` cần làm đường chéo.

In [28]:
def wrong_rolling(s):
    n = len(s)
    dp = [0] * n
    for i in range(n - 1, -1, -1):
        dp[i] = 1
        for j in range(i + 1, n):
            if s[i] == s[j]:
                dp[j] = dp[j - 1] + 2          # SAI: dp[j-1] đã là giá trị mới f[i][j-1], không phải f[i+1][j-1]
            else:
                dp[j] = max(dp[j], dp[j - 1])
    return dp[n - 1]

print("sai :", wrong_rolling("bbbab"), "| đúng:", Solution().longestPalindromeSubseq("bbbab"))

sai : 7 | đúng: 4


**Vì sao sai:** `dp[j-1]` lúc này đã được ghi đè thành `f[i][j-1]`, còn công thức cần `f[i+1][j-1]` (giá trị **cũ** của `dp[j-1]`). **Cách sửa:** dùng biến `prev` cất giá trị cũ trước khi ghi đè (như code cách 4).

### Lỗi 4: tạo bảng bằng `[[0] * n] * n`

In [29]:
def wrong_alias(s):
    n = len(s)
    f = [[0] * n] * n                        # SAI: n dòng là cùng một list
    for i in range(n):
        f[i][i] = 1
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            if s[i] == s[j]:
                f[i][j] = f[i + 1][j - 1] + 2
            else:
                f[i][j] = max(f[i + 1][j], f[i][j - 1])
    return f[0][n - 1]

print("sai :", wrong_alias("bbbab"), "| đúng:", Solution().longestPalindromeSubseq("bbbab"))

sai : 7 | đúng: 4


**Vì sao sai:** mọi `f[i]` là cùng một đối tượng, nên `f[i][j] = ...` ghi vào mọi hàng, phá hỏng các ô khác. **Cách sửa:** `[[0] * n for _ in range(n)]`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra interval DP / LCS:**

- Hỏi về **dãy con** (không liên tiếp) của **một chuỗi**, kèm điều kiện đối xứng ⇒ nghĩ ngay tới `f[i][j]` trên đoạn `[i, j]`: **so hai đầu**, rồi thu hẹp từ hai phía.
- Hai chuỗi, hỏi dãy con chung ⇒ LCS. Palindrome = chuỗi bằng chính đảo của nó ⇒ LCS với `s[::-1]`.
- `n ≤ 1000` gợi ý đáp án cỡ `O(n²)`.

**Một số mẹo cài đặt:**

- Thêm một dòng/cột đệm để khỏi xử lý riêng `i + 1 > n - 1` hay `j - 1 < 0`.
- Nếu muốn tránh việc nhớ thứ tự lặp, lặp theo **độ dài đoạn** `length = 2..n`, rồi `i` từ `0`, `j = i + length - 1`.
- Cách khác: nhớ `f[i][j] = f[i+1][j-1] + 2` chỉ khi `s[i] == s[j]`, còn lại luôn là `max` hai phía.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 5. Longest Palindromic Substring | chuỗi con liên tiếp ⇒ mở rộng từ tâm |
| 647. Palindromic Substrings | đếm palindrome liên tiếp |
| 1312. Min Insertion Steps to Make a String Palindrome | `n - LPS` |
| 1143. Longest Common Subsequence | khung LCS |
| 730. Count Different Palindromic Subsequences | đếm, Hard |
| 132. Palindrome Partitioning II | palindrome + DP cắt |

## 15. Biến thể

### 15.1. Số ký tự cần chèn tối thiểu để thành palindrome (LeetCode 1312)

Các ký tự thuộc LPS đã "tự ghép cặp"; mỗi ký tự **còn lại** cần một ký tự chèn để có cặp. Vậy đáp án là `n - LPS(s)`. Kiểm chứng bằng vét cạn: thử chèn ít nhất `k` ký tự vào mọi vị trí.

In [30]:
def min_insertions(s):
    return len(s) - Solution().longestPalindromeSubseq(s)

def min_insertions_bruteforce(s):
    # BFS theo số lần chèn trên bảng chữ cái của s; chỉ dùng cho chuỗi rất nhỏ
    alpha = sorted(set(s))
    frontier = {s}
    k = 0
    while True:
        if any(t == t[::-1] for t in frontier):
            return k
        nxt = set()
        for t in frontier:
            for pos in range(len(t) + 1):
                for ch in alpha:
                    nxt.add(t[:pos] + ch + t[pos:])
        frontier = nxt
        k += 1

for s in ["zzazz", "mbadm", "leetcode", "ab", "abcd"]:
    a = min_insertions(s)
    if len(s) <= 5:
        assert a == min_insertions_bruteforce(s), s
    print(repr(s), "->", a)

'zzazz' -> 0
'mbadm' -> 2
'leetcode' -> 5
'ab' -> 1
'abcd' -> 3


### 15.2. In ra chính dãy con đối xứng (truy vết)

Sau khi có bảng `f`, bắt đầu từ `(0, n-1)`: nếu hai đầu bằng nhau thì lấy ký tự đó cho hai phía rồi vào `(i+1, j-1)`; nếu khác thì đi về phía có giá trị lớn hơn.

In [31]:
def reconstruct_lps(s):
    n = len(s)
    f = [[0] * (n + 1) for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        f[i][i] = 1
        for j in range(i + 1, n):
            f[i][j] = f[i + 1][j - 1] + 2 if s[i] == s[j] else max(f[i + 1][j], f[i][j - 1])
    left, right = [], []
    i, j = 0, n - 1
    while i <= j:
        if i == j:
            left.append(s[i])
            break
        if s[i] == s[j]:
            left.append(s[i])
            right.append(s[j])
            i += 1
            j -= 1
        elif f[i + 1][j] >= f[i][j - 1]:
            i += 1
        else:
            j -= 1
    return "".join(left) + "".join(reversed(right))

for s in ["bbbab", "cbbd", "agbdba", "character"]:
    p = reconstruct_lps(s)
    assert p == p[::-1] and len(p) == Solution().longestPalindromeSubseq(s)
    print(repr(s), "->", repr(p))

'bbbab' -> 'bbbb'
'cbbd' -> 'bb'
'agbdba' -> 'abdba'
'character' -> 'carac'


### 15.3. Đếm số dãy con đối xứng (đếm theo chỉ số, không loại trùng)

Cho `g[i][j]` = số tập chỉ số (kể cả rỗng) của `s[i..j]` tạo palindrome. Công thức: `g[i][j] = g[i+1][j] + g[i][j-1] - g[i+1][j-1]`, và nếu `s[i] == s[j]` thì cộng thêm `g[i+1][j-1]` (ghép hai đầu với mọi palindrome bên trong, kể cả rỗng). Kiểm bằng vét cạn.

In [32]:
def count_pal_subseq(s):
    n = len(s)
    # g[i][j] đếm cả dãy rỗng: đoạn rỗng (i > j) có 1
    g = [[1] * (n + 2) for _ in range(n + 2)]
    for i in range(n - 1, -1, -1):
        g[i][i] = 2                                   # rỗng và chính ký tự đó
        for j in range(i + 1, n):
            v = g[i + 1][j] + g[i][j - 1] - g[i + 1][j - 1]
            if s[i] == s[j]:
                v += g[i + 1][j - 1]
            g[i][j] = v
    return g[0][n - 1] - 1                            # bỏ dãy rỗng

def count_pal_subseq_bruteforce(s):
    n = len(s)
    c = 0
    for mask in range(1, 1 << n):
        sub = [s[i] for i in range(n) if (mask >> i) & 1]
        c += sub == sub[::-1]
    return c

for s in ["a", "aa", "aba", "bbbab", "abcba", "aaaa"]:
    assert count_pal_subseq(s) == count_pal_subseq_bruteforce(s), s
    print(repr(s), "->", count_pal_subseq(s))

'a' -> 1
'aa' -> 3
'aba' -> 5
'bbbab' -> 19
'abcba' -> 13
'aaaa' -> 15


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử mọi tập chỉ số bằng bitmask, kiểm tra palindrome | `O(n · 2^n)` | `O(n)` | Chỉ để làm oracle kiểm tra (`n ≤ 16`) |
| 2. Memo top-down | Đệ quy trên `[i, j]` + `lru_cache` | `O(n²)` | `O(n²)` + ngăn xếp | Khi dễ nghĩ ra đệ quy, ít ô được ghé tới |
| 3. Bảng 2 chiều | Lặp `i` giảm, `j` tăng, điền `f[i][j]` | `O(n²)` | `O(n²)` | **Nộp bài**: rõ ràng, an toàn, cần truy vết thì có sẵn bảng |
| 4. Một mảng | Giữ hàng `i+1`, cất đường chéo `prev` | `O(n²)` | `O(n)` | Cần tiết kiệm bộ nhớ |
| 5. LCS với `s[::-1]` | Đưa về bài LCS kinh điển | `O(n²)` (gấp ~2 hằng số) | `O(n)` | Khi đã có sẵn hàm LCS; góc nhìn độc lập để kiểm tra |

Đo nhanh thời gian thực tế ở `n = 1000` (số liệu chỉ để so sánh tương đối, mỗi máy một khác).

In [33]:
rng = random.Random(7)
s = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(1000))
ans = None
for name in ["memo", "table", "rolling", "lcs"]:
    t0 = time.perf_counter()
    r = methods[name](s)
    dt = time.perf_counter() - t0
    ans = r if ans is None else ans
    assert r == ans
    print(f"{name:8s} -> {r}  {dt:.3f}s")

memo     -> 309  0.262s
table    -> 309  0.059s
rolling  -> 309  0.049s


lcs      -> 309  0.106s


## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm (hoặc gán `None`), không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, hãy dự đoán kết quả của 4 lời gọi sau, rồi so với đáp án ở mục 19.

```python
Solution().longestPalindromeSubseq("aaaa")
Solution().longestPalindromeSubseq("abcd")
Solution().longestPalindromeSubseq("abca")
Solution().longestPalindromeSubseq("agbcba")
```

### Bài 2 (tính tay)

Với `s = "abacb"` hãy điền bảng `f` (5 x 5, phần `i <= j`) bằng tay theo mục 6, rồi nêu `f[0][4]` và một dãy con đối xứng tương ứng.

### Bài 3 (điền TODO)

Hoàn thành hàm điền bảng dưới đây (chỉ điền hai nhánh của công thức).

In [34]:
def lps_fill_todo(s):
    n = len(s)
    f = [[0] * n for _ in range(n)]
    for i in range(n):
        f[i][i] = 1
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            if s[i] == s[j]:
                pass  # TODO: f[i][j] = ?
            else:
                pass  # TODO: f[i][j] = ?
    return f[0][n - 1]
# Không gọi lps_fill_todo ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết hàm `lps_by_length(s)` bằng **vòng lặp theo độ dài đoạn** `length = 2..n` (với `i` từ `0` đến `n - length`, `j = i + length - 1`). Cho phép dùng bảng 2 chiều.

In [35]:
def lps_by_length(s):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `min_deletions_to_palindrome(s)` trả về **số ký tự tối thiểu cần xóa** để `s` thành palindrome, rồi kiểm chứng với bài vét cạn nhỏ. (Gợi ý: xóa những ký tự không thuộc LPS.)

In [36]:
def min_deletions_to_palindrome(s):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: một dãy con đối xứng dài nhất của `"aaaa"` là gì? Và của `"abcd"` toàn khác nhau?
- Mức 2: với `"abca"` hai đầu bằng nhau, vậy là `2 +` đáp án của `"bc"`.
- Mức 3: `"agbcba"`: hai đầu `a`-`a`, bên trong `"gbcb"` có `"bcb"` dài 3; tổng là 5.

**Bài 2**

- Mức 1: đường chéo `f[i][i] = 1`; ô `(i, i+1)` bằng `2` nếu hai chữ bằng nhau, ngược lại `1`.
- Mức 2: điền từng hàng, từ hàng `i = 4` ngược lên `0`.
- Mức 3: với `s = "abacb"`, đáp án là 3 (ví dụ `"aba"` từ chỉ số 0,1,2, hoặc `"bab"` từ chỉ số 1,2,4); hãy kiểm lại bằng bảng của bạn.

**Bài 3**

- Mức 1: nhánh bằng nhau là "ăn cả hai đầu".
- Mức 2: nhánh bằng nhau dùng ô đường chéo trên-trái `f[i+1][j-1]`.
- Mức 3: `f[i][j] = f[i+1][j-1] + 2` và `f[i][j] = max(f[i+1][j], f[i][j-1])`.

**Bài 4**

- Mức 1: đoạn dài 1 có đáp án 1; hãy xử lý theo độ dài tăng dần để ô cần dùng luôn sẵn.
- Mức 2: với `length >= 2`: `j = i + length - 1`; ô `(i+1, j-1)` khi `length = 2` là đoạn rỗng.
- Mức 3: khởi tạo `f[i][i] = 1`, còn đoạn rỗng tra `0` (bảng đệm hoặc kiểm `i + 1 <= j - 1`).

**Bài 5**

- Mức 1: các ký tự thuộc LPS thì giữ lại.
- Mức 2: số ký tự cần xóa = `n - LPS`.
- Mức 3: `return len(s) - Solution().longestPalindromeSubseq(s)`; để kiểm, thử mọi tập xóa bằng bitmask.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
assert Solution().longestPalindromeSubseq("aaaa") == 4
assert Solution().longestPalindromeSubseq("abcd") == 1
assert Solution().longestPalindromeSubseq("abca") == 3
assert Solution().longestPalindromeSubseq("agbcba") == 5
print("Bài 1: 4, 1, 3, 5")

# Bài 2: bảng tay cho "abacb"
s = "abacb"
f = f_table(s)
n = len(s)
for i in range(n):
    print(f"{i}:{s[i]}", [f[i][j] if j >= i else "." for j in range(n)])
assert f[0][4] == 3
print("Bài 2: f[0][4] =", f[0][4], "| một dãy con:", reconstruct_lps(s))

Bài 1: 4, 1, 3, 5
0:a [1, 1, 3, 3, 3]
1:b ['.', 1, 1, 1, 3]
2:a ['.', '.', 1, 1, 1]
3:c ['.', '.', '.', 1, 1]
4:b ['.', '.', '.', '.', 1]
Bài 2: f[0][4] = 3 | một dãy con: bcb


In [38]:
# Bài 3
def lps_fill_answer(s):
    n = len(s)
    f = [[0] * n for _ in range(n)]
    for i in range(n):
        f[i][i] = 1
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            if s[i] == s[j]:
                f[i][j] = f[i + 1][j - 1] + 2
            else:
                f[i][j] = max(f[i + 1][j], f[i][j - 1])
    return f[0][n - 1]

# Bài 4
def lps_by_length_answer(s):
    n = len(s)
    f = [[0] * (n + 1) for _ in range(n + 1)]       # đệm để f[i+1][j-1] luôn hợp lệ
    for i in range(n):
        f[i][i] = 1
    for length in range(2, n + 1):
        for i in range(0, n - length + 1):
            j = i + length - 1
            if s[i] == s[j]:
                f[i][j] = (f[i + 1][j - 1] if i + 1 <= j - 1 else 0) + 2
            else:
                f[i][j] = max(f[i + 1][j], f[i][j - 1])
    return f[0][n - 1]

# Bài 5
def min_deletions_answer(s):
    return len(s) - Solution().longestPalindromeSubseq(s)

def min_deletions_bruteforce(s):
    n = len(s)
    best = n
    for mask in range(1 << n):                      # mask = tập ký tự GIỮ LẠI
        sub = [s[i] for i in range(n) if (mask >> i) & 1]
        if sub == sub[::-1]:
            best = min(best, n - len(sub))
    return best

rng = random.Random(99)
for _ in range(200):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    expected = Solution().longestPalindromeSubseq(s)
    assert lps_fill_answer(s) == expected
    assert lps_by_length_answer(s) == expected
    assert min_deletions_answer(s) == min_deletions_bruteforce(s)
print("Bài 3, 4, 5: OK")

Bài 3, 4, 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (bảng 2 chiều):**

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        f = [[0] * n for _ in range(n)]          # f[i][j]: đáp án của s[i..j]
        for i in range(n):
            f[i][i] = 1                           # 1 ký tự
        for i in range(n - 1, -1, -1):            # i giảm: hàng dưới có trước
            for j in range(i + 1, n):             # j tăng: cột trái có trước
                if s[i] == s[j]:
                    f[i][j] = f[i + 1][j - 1] + 2         # ăn cả hai đầu
                else:
                    f[i][j] = max(f[i + 1][j], f[i][j - 1])  # bỏ một đầu
        return f[0][n - 1]
```

**Câu thần chú:** *"Hai đầu bằng nhau thì ăn cả hai, khác nhau thì bỏ một đầu; hàng dưới trước, cột trái trước."*

**Checklist trước khi submit:**

- [ ] Đọc kỹ: là **dãy con** (subsequence), không phải chuỗi con liên tiếp.
- [ ] Đề hỏi **độ dài** (không phải chính chuỗi).
- [ ] `f[i][i] = 1` đã đặt; ô dưới đường chéo là `0` (đoạn rỗng).
- [ ] Thứ tự lặp: `i` từ `n-1` xuống `0`, `j` từ `i+1` lên `n-1`.
- [ ] Bảng tạo bằng `[[0] * n for _ in range(n)]`, không phải `[[0] * n] * n`.
- [ ] Nếu nén một mảng: đã cất đường chéo `prev` trước khi ghi đè.
- [ ] Đã thử `n = 1`, chuỗi toàn giống nhau, toàn khác nhau, chuỗi đã là palindrome.
- [ ] Với đệ quy: nhớ `@cache` và cân nhắc giới hạn đệ quy (`n ≤ 1000`).
- [ ] Độ phức tạp `O(n²)` thời gian là đủ cho `n ≤ 1000`.